# 02 - Limpieza
Aplicacion de las reglas documentadas en docs/methodology.md sobre el subconjunto
de suicidio 2023 (9,072 registros, ya filtrado en 01_profiling.ipynb).

Reglas a aplicar:
1. Recodificar codigos de 'no especificado'/'se ignora' a NaN explicito (no se imputa)
2. Separar la variable Edad (mezcla unidad + valor)
3. Traducir catalogos clave a etiquetas legibles (Sexo, causa CIE-10, entidad)
4. Verificar resultados y guardar en data/processed/

In [ ]:
import pandas as pd
import os
import sys
sys.path.append('../src')
from cleaning_utils import (
    load_dbf, normalize_columns, CATALOG_CANONICAL_COLUMNS, null_summary, dtype_summary,
    recode_null_codes, split_edad, translate_catalog
)


## 1. Cargar el subconjunto filtrado (handoff de 01_profiling.ipynb)

In [ ]:
df = pd.read_csv('../data/interim/suicidio_2023_filtrado.csv', encoding='utf-8', low_memory=False, dtype=str)
print(f'Registros cargados: {len(df):,}')
df.head()


## 2. Cargar catalogos necesarios para traducir etiquetas

In [ ]:
cat_geo = load_dbf('../data/raw/CATEMLDE23.dbf')
cat_geo = normalize_columns(cat_geo, canonical_names=CATALOG_CANONICAL_COLUMNS)
cat_causa = load_dbf('../data/raw/CATMINDE.dbf')
cat_causa = normalize_columns(cat_causa, canonical_names=CATALOG_CANONICAL_COLUMNS)
print('Catalogo geografico:', cat_geo.shape)
print('Catalogo causa CIE-10:', cat_causa.shape)
cat_geo.head()


## 3. Regla 1 - Recodificar 'no especificado'/'se ignora' a NaN explicito
Decision documentada en methodology.md: NO se imputa, se preserva el hueco como NaN real.

In [ ]:
print('Nulos ANTES de recodificar:')
print(null_summary(df).head(10))

df = recode_null_codes(df)

print('\nNulos DESPUES de recodificar:')
print(null_summary(df).head(10))


## 4. Regla 2 - Separar la variable Edad (unidad + valor)
Edad mezcla horas/dias/meses/anios en un solo codigo numerico. Se crean edad_valor y edad_unidad,
conservando la columna original Edad sin modificar.

In [ ]:
df = split_edad(df, col='Edad')
df[['Edad', 'edad_valor', 'edad_unidad']].head(10)


In [ ]:
df['edad_unidad'].value_counts(dropna=False)


## 5. Regla 3 - Traducir catalogos a etiquetas legibles
Se agregan columnas nuevas descriptivas; se conservan los codigos originales.

In [ ]:
# Sexo: etiqueta simple, no requiere catalogo externo
df['sexo_desc'] = df['Sexo'].map({'1': 'Hombre', '2': 'Mujer'})
df['sexo_desc'].value_counts(dropna=False)


In [ ]:
# Causa de defuncion (CIE-10 detallado) via catalogo CATMINDE
df = translate_catalog(df, df_col='Causa_def', catalog_df=cat_causa,
                        catalog_code_col='Cve', catalog_desc_col='Descrip',
                        new_col_name='causa_def_desc')
df[['Causa_def', 'causa_def_desc']].drop_duplicates().head(10)


In [ ]:
# Entidad de ocurrencia via catalogo geografico
# El catalogo geografico combina entidad+municipio+localidad en Cve_ent+Cve_mun+Cve_loc;
# para el nombre de la ENTIDAD, filtramos donde Cve_mun == '000' y Cve_loc == '0000'
cat_entidades = cat_geo[(cat_geo['Cve_mun'] == '000') & (cat_geo['Cve_loc'] == '0000')]
df = translate_catalog(df, df_col='Ent_ocurr', catalog_df=cat_entidades,
                        catalog_code_col='Cve_ent', catalog_desc_col='Nom_loc',
                        new_col_name='ent_ocurr_desc')
df[['Ent_ocurr', 'ent_ocurr_desc']].drop_duplicates().sort_values('Ent_ocurr').head(15)


## 6. Verificacion antes de guardar
Confirmar que el volumen de registros no cambio (la limpieza no debe filtrar filas, solo corregir valores).

In [ ]:
print(f'Registros: {len(df):,} (debe seguir siendo 9,072)')
print(f'Columnas: {df.shape[1]} (74 originales + nuevas: edad_valor, edad_unidad, sexo_desc, causa_def_desc, ent_ocurr_desc)')
df.head()


## 7. Guardar dataset limpio en data/processed/

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/suicidio_2023_limpio.csv', index=False, encoding='utf-8')
print(f'Guardado: {len(df):,} registros x {df.shape[1]} columnas en data/processed/suicidio_2023_limpio.csv')


## 8. Hallazgos y decisiones de la fase de limpieza
_Documentar aqui: cuantos valores se recodificaron por columna, distribucion de edad_unidad,
cualquier caso inesperado encontrado. Trasladar a docs/methodology.md y docs/quality_report.md._

## 9. Visualizaciones exploratorias
Graficas de apoyo para el portafolio: distribucion por sexo, edad, causa de defuncion y entidad.
Se guardan como PNG en docs/figuras/ para poder referenciarlas en el README o el quality_report.

In [ ]:
import matplotlib.pyplot as plt
import os
os.makedirs('../docs/figuras', exist_ok=True)
plt.rcParams['figure.dpi'] = 110


### 9.1 Casos por sexo

In [ ]:
conteo_sexo = df['sexo_desc'].value_counts(dropna=False)
fig, ax = plt.subplots(figsize=(5,4))
conteo_sexo.plot(kind='bar', color=['#4C72B0','#DD8452'], ax=ax)
ax.set_title('Casos de suicidio 2023 por sexo')
ax.set_xlabel('')
ax.set_ylabel('Numero de casos')
plt.xticks(rotation=0)
for i, v in enumerate(conteo_sexo):
    ax.text(i, v + 30, f'{v:,}', ha='center')
plt.tight_layout()
plt.savefig('../docs/figuras/01_casos_por_sexo.png')
plt.show()
conteo_sexo


### 9.2 Distribucion de edad (solo casos con edad en anios)

In [ ]:
edad_anios = df.loc[df['edad_unidad'] == 'anios', 'edad_valor'].dropna().astype(int)
print(f'Casos con edad en anios: {len(edad_anios):,} de {len(df):,} totales')
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(edad_anios, bins=20, color='#4C72B0', edgecolor='white')
ax.set_title('Distribucion de edad - casos de suicidio 2023')
ax.set_xlabel('Edad (anios)')
ax.set_ylabel('Numero de casos')
plt.tight_layout()
plt.savefig('../docs/figuras/02_distribucion_edad.png')
plt.show()
edad_anios.describe()


### 9.3 Top 10 causas de defuncion (CIE-10 detallado)

In [ ]:
top_causas = df['causa_def_desc'].value_counts(dropna=False).head(10)
fig, ax = plt.subplots(figsize=(8,5))
top_causas.sort_values().plot(kind='barh', color='#55A868', ax=ax)
ax.set_title('Top 10 causas de defuncion (CIE-10) - suicidio 2023')
ax.set_xlabel('Numero de casos')
plt.tight_layout()
plt.savefig('../docs/figuras/03_top_causas.png')
plt.show()
top_causas


### 9.4 Casos por entidad de ocurrencia (estado)

In [ ]:
casos_entidad = df['ent_ocurr_desc'].value_counts(dropna=False)
fig, ax = plt.subplots(figsize=(8,10))
casos_entidad.sort_values().plot(kind='barh', color='#C44E52', ax=ax)
ax.set_title('Casos de suicidio 2023 por entidad de ocurrencia')
ax.set_xlabel('Numero de casos')
plt.tight_layout()
plt.savefig('../docs/figuras/04_casos_por_entidad.png')
plt.show()
casos_entidad


### 9.5 Casos por mes de ocurrencia (estacionalidad)

In [ ]:
meses_labels = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
                7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}
df['Mes_ocurr_num'] = pd.to_numeric(df['Mes_ocurr'], errors='coerce')
casos_mes = df['Mes_ocurr_num'].map(meses_labels).value_counts().reindex(list(meses_labels.values()))
fig, ax = plt.subplots(figsize=(8,4))
casos_mes.plot(kind='bar', color='#8172B2', ax=ax)
ax.set_title('Casos de suicidio 2023 por mes de ocurrencia')
ax.set_xlabel('')
ax.set_ylabel('Numero de casos')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../docs/figuras/05_casos_por_mes.png')
plt.show()
casos_mes


### 9.6 Casos por dia de la semana
Reconstruido a partir de Anio_ocur + Mes_ocurr + Dia_ocurr.

In [ ]:
fechas_ocurr = pd.to_datetime({
    'year': pd.to_numeric(df['Anio_ocur'], errors='coerce'),
    'month': pd.to_numeric(df['Mes_ocurr'], errors='coerce'),
    'day': pd.to_numeric(df['Dia_ocurr'], errors='coerce'),
}, errors='coerce')
dias_es = {'Monday':'Lunes','Tuesday':'Martes','Wednesday':'Miercoles','Thursday':'Jueves',
           'Friday':'Viernes','Saturday':'Sabado','Sunday':'Domingo'}
df['dia_semana'] = fechas_ocurr.dt.day_name().map(dias_es)
orden_dias = ['Lunes','Martes','Miercoles','Jueves','Viernes','Sabado','Domingo']
casos_dia = df['dia_semana'].value_counts().reindex(orden_dias)
print(f'Fechas invalidas/no reconstruibles: {fechas_ocurr.isna().sum():,}')
fig, ax = plt.subplots(figsize=(8,4))
casos_dia.plot(kind='bar', color='#64B5CD', ax=ax)
ax.set_title('Casos de suicidio 2023 por dia de la semana')
ax.set_xlabel('')
ax.set_ylabel('Numero de casos')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../docs/figuras/06_casos_por_dia_semana.png')
plt.show()
casos_dia


### 9.7 Grupo de edad (quinquenal oficial INEGI) cruzado con sexo
Mas interpretable que el histograma de edad continua: usa las categorias oficiales de Edad_agru.

In [ ]:
edad_agru_labels = {
    '01':'Menores de 1 año','02':'1 año','03':'2 años','04':'3 años','05':'4 años',
    '06':'5 a 9','07':'10 a 14','08':'15 a 19','09':'20 a 24','10':'25 a 29',
    '11':'30 a 34','12':'35 a 39','13':'40 a 44','14':'45 a 49','15':'50 a 54',
    '16':'55 a 59','17':'60 a 64','18':'65 a 69','19':'70 a 74','20':'75 a 79',
    '21':'80 a 84','22':'85 a 89','23':'90 a 94','24':'95 a 99',
    '25':'100 a 104','26':'105 a 109','27':'110 a 114','28':'115 a 119','29':'120 y mas',
}
df['edad_agru_desc'] = df['Edad_agru'].astype(str).str.zfill(2).map(edad_agru_labels)
orden_edades = [v for k, v in sorted(edad_agru_labels.items())]
tabla_edad_sexo = pd.crosstab(df['edad_agru_desc'], df['sexo_desc']).reindex(orden_edades).fillna(0)
# limitar a rangos con casos, para no saturar la grafica con grupos en 0
tabla_edad_sexo = tabla_edad_sexo[tabla_edad_sexo.sum(axis=1) > 0]
fig, ax = plt.subplots(figsize=(8,7))
tabla_edad_sexo.plot(kind='barh', stacked=True, color=['#4C72B0','#DD8452'], ax=ax)
ax.set_title('Casos de suicidio 2023 por grupo de edad y sexo')
ax.set_xlabel('Numero de casos')
ax.set_ylabel('')
ax.legend(title='Sexo')
plt.tight_layout()
plt.savefig('../docs/figuras/07_edad_grupo_por_sexo.png')
plt.show()
tabla_edad_sexo


### 9.8 Area urbana vs rural

In [ ]:
area_labels = {'1':'Urbana', '2':'Rural'}
df['area_desc'] = df['Area_ur'].astype(str).str.strip().map(area_labels)
casos_area = df['area_desc'].value_counts(dropna=False)
fig, ax = plt.subplots(figsize=(5,5))
colores = ['#55A868','#C44E52','#8C8C8C']
ax.pie(casos_area, labels=casos_area.index.astype(str), autopct='%1.1f%%',
       colors=colores[:len(casos_area)])
ax.set_title('Casos de suicidio 2023: area urbana vs rural')
plt.tight_layout()
plt.savefig('../docs/figuras/08_area_urbana_rural.png')
plt.show()
casos_area
